# Notebook Goal
In this notebook I will try to add phone attritibute in our drivers schema and test auto loader.

```text
new JSON with phone
        │
        ▼
Auto Loader detects change
        │
        ▼
source schema evolves
        │
        ▼
Bronze Delta schema evolves
        │
        ▼
old rows: phone = NULL
new rows: phone = actual value
        │
        ▼
Silver intentionally adopts phone
```
Schema evoluation happens in two step: Auto Loader (Source Schema) & Delta Table (target schema)

## Current Schema
There is no phone

In [0]:
spark.table(
    "ride_lakehouse_dev.bronze.drivers"
).printSchema()

root
 |-- car_number: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- generated_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



Silver almost same with bronze schema -, the only difference is timestamp for silver transform (Bronze->Silver)

In [0]:
spark.table(
    "ride_lakehouse_dev.silver.drivers"
).printSchema()

root
 |-- car_number: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- generated_at: timestamp (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



# New data
We will use faker library again. And only driver ath this time

In [0]:
from pyspark.sql import functions as F

In [0]:
%pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.8 MB/s  0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from faker import Faker
from datetime import datetime, timezone
import random
import json

fake = Faker()

BASE_PATH = (
    "/Volumes/"
    "ride_lakehouse_dev/"
    "bronze/"
    "streaming_files"
)

DRIVERS_LANDING = (
    f"{BASE_PATH}/landing/drivers"
)

DRIVERS_SCHEMA = (
    f"{BASE_PATH}/schemas/drivers"
)

DRIVERS_CHECKPOINT = (
    f"{BASE_PATH}/checkpoints/bronze_drivers"
)

## New driver schema

In [0]:
def generate_drivers_v2(
    batch_id,
    count=10
):

    drivers = []

    for i in range(count):

        driver = {
            "id": f"D-{batch_id}-{i:03d}",
            "name": fake.name(),
            "car_number": fake.bothify(
                text="??-####"
            ).upper(),
            "experience": random.randint(
                1,
                20
            ),
            "rating": round(
                random.uniform(3.0, 5.0),
                2
            ),

            # new column
            "phone": fake.phone_number(),

            "generated_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            )
        }

        drivers.append(driver)

    return drivers

In [0]:
batch_id = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%d%H%M%S%f")
)

In [0]:
drivers_v2 = generate_drivers_v2(
    batch_id=batch_id,
    count=10
)

In [0]:
drivers_v2[:2]

[{'id': 'D-20260930101247834142-000',
  'name': 'Eileen Ortiz',
  'car_number': 'FC-4992',
  'experience': 19,
  'rating': 3.33,
  'phone': '001-703-791-6210',
  'generated_at': '2026-09-30T10:13:28.394402+00:00'},
 {'id': 'D-20260930101247834142-001',
  'name': 'Stephen Rice',
  'car_number': 'CS-5438',
  'experience': 3,
  'rating': 4.57,
  'phone': '+1-242-870-6417',
  'generated_at': '2026-09-30T10:13:28.394607+00:00'}]

# Write new file
We will send new format data to volume in jsson format

In [0]:
file_path = (
    f"{DRIVERS_LANDING}/"
    f"drivers_schema_v2_{batch_id}.json"
)

json_content = "\n".join(
    json.dumps(record)
    for record in drivers_v2
)

dbutils.fs.put(
    file_path,
    json_content,
    overwrite=False
)

print(file_path)

Wrote 2016 bytes.
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json


At this moment nothing happened yet because we just send file to volume. Auto Loader didn't reconize yet. We need to read data in volume

# Bronze reader
Now we will read data and thanks to the chec point auto loader read only new file and reconize new schema

In [0]:
from pyspark.sql import functions as F

drivers_stream = (
    spark.readStream
        .format("cloudFiles")

        .option(
            "cloudFiles.format",
            "json"
        )

        .option(
            "cloudFiles.schemaLocation",
            DRIVERS_SCHEMA
        )

        .option(
            "cloudFiles.inferColumnTypes",
            "true"
        )

        .option(
            "cloudFiles.schemaEvolutionMode",
            "addNewColumns"
        )

        .load(
            DRIVERS_LANDING
        )

        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )

        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

In [0]:
drivers_query = (
    drivers_stream
        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            DRIVERS_CHECKPOINT
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .toTable(
            "ride_lakehouse_dev.bronze.drivers"
        )
)



After restaring we need to run write query code again so delta can update it schema with merge
```text
Auto Loader schema
now knows phone
       │
       ▼
Streaming DataFrame
contains phone
       │
       ▼
Delta target
doesn't yet contain phone
       │
       ▼
mergeSchema = true
       │
       ▼
Delta table adds phone
```

In [0]:
drivers_stream.printSchema()

root
 |-- car_number: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- generated_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- phone: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)



Now we can see phone

In [0]:
%sql
SELECT
    id,
    name,
    phone,
    generated_at,
    _source_file

FROM ride_lakehouse_dev.bronze.drivers

ORDER BY _ingested_at DESC;

id,name,phone,generated_at,_source_file
D-20260930101247834142-005,Daniel Sanchez,(481)819-5811x698,2026-09-30T10:13:28.395456+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-008,Dustin Garrett,718.934.7024x021,2026-09-30T10:13:28.395939+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-004,Bryce Barnett,001-883-778-6778x994,2026-09-30T10:13:28.395273+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-000,Eileen Ortiz,001-703-791-6210,2026-09-30T10:13:28.394402+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-003,Kristen Ford,553.342.5219x8678,2026-09-30T10:13:28.395085+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-001,Stephen Rice,+1-242-870-6417,2026-09-30T10:13:28.394607+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-007,Hunter Morris,362-603-3480x32437,2026-09-30T10:13:28.395773+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-006,Jay Gomez,636.753.3342x899,2026-09-30T10:13:28.395619+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-009,Whitney Mcdonald,988.290.7455x33414,2026-09-30T10:13:28.396090+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json
D-20260930101247834142-002,Susan Sosa,001-963-677-9232,2026-09-30T10:13:28.394762+00:00,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_schema_v2_20260930101247834142.json


In [0]:
%sql
DESCRIBE HISTORY
ride_lakehouse_dev.bronze.drivers;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-09-30T10:24:27.000Z,77111473424338,dogancandemir2001@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 677caa46-d476-48de-a45d-97ad6ff9e6ef, epochId -> 1, statsOnLoad -> false)",null,List(3795482916102374),61953070-7830-40f1-ba00-be3c40824a49,0930-100558-xuyvpnk4-v2n,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 10, numOutputBytes -> 4159, numAddedFiles -> 1)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
2,2026-09-30T10:24:20.000Z,77111473424338,dogancandemir2001@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 677caa46-d476-48de-a45d-97ad6ff9e6ef, epochId -> -1, statsOnLoad -> false)",null,List(3795482916102374),61953070-7830-40f1-ba00-be3c40824a49,0930-100558-xuyvpnk4-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-29T13:01:34.000Z,77111473424338,dogancandemir2001@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 677caa46-d476-48de-a45d-97ad6ff9e6ef, epochId -> 0, statsOnLoad -> true)",null,List(2906593542381095),df9edd17-c3e5-4178-8c7a-4bab9b89a780,0929-124122-enfn1tx2-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 40, numOutputBytes -> 4611, numAddedFiles -> 1)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-29T13:01:21.000Z,77111473424338,dogancandemir2001@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""9aa40f48-4bb4-4d8b-b52d-d20ae6e77456"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6607b287-ad52-405b-9a1b-5e051e1f6ebd"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-98f2e2ff-855b-41f7-936b-87594abc74aa""}, statsOnLoad -> false)",null,List(2906593542381095),df9edd17-c3e5-4178-8c7a-4bab9b89a780,0929-124122-enfn1tx2-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT
    id,
    name,
    phone

FROM ride_lakehouse_dev.bronze.drivers

WHERE phone IS NOT NULL

LIMIT 20;

id,name,phone
D-20260930101247834142-000,Eileen Ortiz,001-703-791-6210
D-20260930101247834142-001,Stephen Rice,+1-242-870-6417
D-20260930101247834142-002,Susan Sosa,001-963-677-9232
D-20260930101247834142-003,Kristen Ford,553.342.5219x8678
D-20260930101247834142-004,Bryce Barnett,001-883-778-6778x994
D-20260930101247834142-005,Daniel Sanchez,(481)819-5811x698
D-20260930101247834142-006,Jay Gomez,636.753.3342x899
D-20260930101247834142-007,Hunter Morris,362-603-3480x32437
D-20260930101247834142-008,Dustin Garrett,718.934.7024x021
D-20260930101247834142-009,Whitney Mcdonald,988.290.7455x33414


# Silver Layer Evoluation

In [0]:
SILVER_DRIVERS_CHECKPOINT = (
    f"{BASE_PATH}/checkpoints/silver_drivers"
)

In [0]:
bronze_drivers_stream = (
    spark.readStream
        .table(
            "ride_lakehouse_dev.bronze.drivers"
        )
)

In [0]:
bronze_drivers_stream.printSchema()

root
 |-- car_number: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- generated_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- phone: string (nullable = true)



we can see phone on bronze layer

Updated Silver driver transformation:

In [0]:
silver_drivers_stream = (
    bronze_drivers_stream

        .withColumn(
            "generated_at",
            F.to_timestamp("generated_at")
        )

        .withColumn(
            "name",
            F.trim(F.col("name"))
        )

        .withColumn(
            "car_number",
            F.upper(
                F.trim(F.col("car_number"))
            )
        )

        .filter(
            F.col("id").isNotNull()
            &
            (F.trim(F.col("id")) != "")
        )

        .filter(
            F.col("name").isNotNull()
        )

        .filter(
            F.col("car_number").isNotNull()
        )

        .filter(
            F.col("rating").between(0.0, 5.0)
        )

        .filter(
            F.col("experience").between(0, 60)
        )

        .dropDuplicates(["id"])

        .withColumn( # New data
            "phone",
            F.trim(
                F.col("phone")
            )
        )

        .withColumn(
            "_silver_processed_at",
            F.current_timestamp()
        )
)

In [0]:
silver_drivers_query = (
    silver_drivers_stream
        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            SILVER_DRIVERS_CHECKPOINT
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .toTable(
            "ride_lakehouse_dev.silver.drivers"
        )
)

silver_drivers_query.awaitTermination()

In [0]:
spark.table(
    "ride_lakehouse_dev.silver.drivers"
).printSchema()

root
 |-- car_number: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- generated_at: timestamp (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)
 |-- phone: string (nullable = true)

